## **Task Title: Fine-tune a transformer model (BERT/DistilBERT) to perform: Part-of-Speech (POS) Tagging- Chunking**


In [1]:
!pip install datasets==2.19.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 14.8 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.3.1 which is incompatible.


In [2]:
!pip install conllu

In [ ]:
!pip install -q transformers datasets evaluate seqeval

In [7]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification, Trainer, TrainingArguments
import numpy as np
from evaluate import load

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

import datasets
import os

# Get the cache directory for dataset builders
builder_cache_dir = os.path.join(datasets.config.HF_DATASETS_CACHE, "universal_dependencies")
# Construct the path to the universal_dependencies builder cache
ud_cache_path = builder_cache_dir

# Check if the cache directory exists and remove it if it does
if os.path.exists(ud_cache_path):
    print(f"Deleting old universal_dependencies cache at: {ud_cache_path}")
    import shutil
    shutil.rmtree(ud_cache_path)
    print("Cache deleted. Attempting to reload dataset.")

dataset = load_dataset("universal_dependencies", "en_ewt")

labels = dataset["train"].features["upos"].feature.names
id2label = {i: l for i, l in enumerate(labels)}
label2id = {l: i for i, l in enumerate(labels)}

Deleting old universal_dependencies cache at: /root/.cache/huggingface/datasets/universal_dependencies
Cache deleted. Attempting to reload dataset.


/usr/local/lib/python3.12/dist-packages/datasets/load.py:1486: FutureWarning: The repository for universal_dependencies contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/universal_dependencies
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(


Generating train split:   0%|          | 0/12543 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2002 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2077 [00:00<?, ? examples/s]

In [8]:
# Loading Chunking & POS Model

chunk_dataset = load_dataset("conll2000")
chunk_labels = chunk_dataset["train"].features["chunk_tags"].feature.names

chunk_id2label = {i: l for i, l in enumerate(chunk_labels)}
chunk_label2id = {l: i for i, l in enumerate(chunk_labels)}

chunk_model = AutoModelForTokenClassification.from_pretrained(
    "dslim/bert-base-NER"
)

pos_model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id
)

pos_model.to(device)
chunk_model.to(device)

/usr/local/lib/python3.12/dist-packages/datasets/load.py:1486: FutureWarning: The repository for conll2000 contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/conll2000
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BertForTokenClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, el

In [9]:

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased", padding=True, truncation=True)

# Tokenize + align labels
def tokenize_and_align_labels(examples):
    tokenized = tokenizer(
    examples["tokens"],
    padding="max_length",
    truncation=True,
    is_split_into_words=True,
    max_length=128
   )
    aligned_labels = []

    for i, label_seq in enumerate(examples["upos"]):
        word_ids = tokenized.word_ids(batch_index=i)
        temp = []
        for word_idx in word_ids:
            if word_idx is None:
                temp.append(-100)
            else:
                temp.append(label_seq[word_idx])
        aligned_labels.append(temp)

    tokenized["labels"] = aligned_labels
    return tokenized

tokenized_data = dataset.map(tokenize_and_align_labels, batched=True)
tokenized_data = tokenized_data.remove_columns(
    [col for col in tokenized_data["train"].column_names if col not in ["input_ids", "attention_mask", "labels"]]
)

Map:   0%|          | 0/12543 [00:00<?, ? examples/s]

Map:   0%|          | 0/2002 [00:00<?, ? examples/s]

Map:   0%|          | 0/2077 [00:00<?, ? examples/s]

In [10]:

# Metrics
metric = load("seqeval")

def compute_metrics(p):
    preds, labels_true = p
    preds = np.argmax(preds, axis=2)

    true_labels = [[labels[l] for l in lab if l != -100] for lab in labels_true]
    true_preds = [[labels[p] for p, l in zip(pred_seq, lab) if l != -100]
                  for pred_seq, lab in zip(preds, labels_true)]

    res = metric.compute(predictions=true_preds, references=true_labels)
    return {"f1": res["overall_f1"]}

In [11]:
# Training
args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=8,
    num_train_epochs=2,
    logging_steps=50,
    save_strategy="no",
    remove_unused_columns=False
)

from transformers import DataCollatorForTokenClassification
data_collator = DataCollatorForTokenClassification(tokenizer)


trainer = Trainer(
    model=pos_model,
    args=args,
    train_dataset=tokenized_data["train"],
    eval_dataset=tokenized_data["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

Step,Training Loss
50,1.715476
100,0.477653
150,0.341464
200,0.244075
250,0.206665
300,0.206851
350,0.173914
400,0.188589
450,0.181769
500,0.153035


TrainOutput(global_step=3136, training_loss=0.1362344105252806, metrics={'train_runtime': 343.1609, 'train_samples_per_second': 73.103, 'train_steps_per_second': 9.139, 'total_flos': 819627966305280.0, 'train_loss': 0.1362344105252806, 'epoch': 2.0})

In [6]:
# Inference

sentence = "John works at Google in California"
tokens = sentence.split()

inputs = tokenizer(tokens, is_split_into_words=True, return_tensors="pt")
inputs = {k: v.to(device) for k, v in inputs.items()}

# POS
pos_outputs = pos_model(**inputs).logits
pos_preds = pos_outputs.argmax(-1).squeeze().tolist()

# CHUNK
chunk_results = chunk_pipeline(sentence)

chunk_dict = {item["word"]: item["entity_group"] for item in chunk_results}

print("\nInput:", sentence)
print("\nToken\t\tPOS Tag\t\tChunk Tag")
print("-" * 40)

for token, p in zip(tokens, pos_preds):
    chunk_tag = chunk_dict.get(token, "O")
    print(f"{token}\t\t{labels[p]}\t\t{chunk_tag}")


Input: John works at Google in California

Token		POS Tag		Chunk Tag
----------------------------------------
John		PUNCT		PER
works		PROPN		O
at		VERB		O
Google		ADP		ORG
in		PROPN		O
California		ADP		LOC


# **Comparison :**

| Feature    | POS Tagging            | Chunking                |
| ---------- | ---------------------- | ----------------------- |
| Level      | Word-level             | Phrase-level            |
| Complexity | Easy                   | Medium                  |
| Output     | Tags (NN, VB, JJ)      | Phrases (NP, VP)        |
| Purpose    | Grammar identification | Structure understanding |
| Example    | John → Noun            | [John] → Noun Phrase    |
| Usage      | Basic NLP tasks        | Advanced NLP tasks      |


# **Differences between POS Tagging and Chunking :**


POS tagging and chunking are fundamental NLP techniques used for text analysis. POS tagging assigns grammatical labels such as noun, verb, or adjective to individual words, whereas chunking groups these tagged words into higher-level structures like noun phrases and verb phrases. While POS tagging operates at a basic level, chunking provides deeper insights into sentence structure and meaning.

# **Challenges Faced :**


* Understanding difference between word-level and phrase-level processing.
* Handling ambiguous words (e.g., “book” can be noun or verb).
* Errors propagation: wrong POS → wrong chunking.
* Dataset limitations and preprocessing complexity.
* Model misclassification in real-world sentences.

# **Observations and Insights :**


* POS tagging is easier and faster but limited in context understanding.
* Chunking improves semantic understanding by grouping related words.
* Accurate POS tagging is essential for effective chunking.
* Real-world NLP tasks require combining both techniques.
* Model performance depends heavily on data quality and size.